In [ ]:
# First, ensure you have the libraries installed:
# pip install pandas openpyxl

import pandas as pd
import openpyxl

# The path to your Excel file
file_path = 'sample_docs/sample_statement.xlsx'

# Load the workbook once with openpyxl
try:
    workbook = openpyxl.load_workbook(file_path, data_only=True) # data_only=True reads cell values, not formulas
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
    exit()

#---------------------------------------------------------------------
# Function 1: Extracting from the "Notes" Sheet (Prose/Text)
# Corresponds to: Screenshot 2025-10-13 at 7.23.31 PM.jpg
#---------------------------------------------------------------------
def extract_notes_sheet(sheet):
    """
    This sheet is like a document. We iterate through rows and extract text
    from the main column (looks like column B).
    """
    print("--- Extracting from Notes Sheet ---")
    full_text = []
    # Assuming text is primarily in column B from row 2 onwards
    for row in sheet.iter_rows(min_row=2, min_col=2, max_col=2):
        for cell in row:
            if cell.value: # Check if the cell is not empty
                # .strip() removes leading/trailing whitespace
                full_text.append(str(cell.value).strip())
    
    # Join all text blocks into a single string with newlines
    return "\n".join(full_text)

#---------------------------------------------------------------------
# Function 2: Extracting from a Vertical Financial Report
# Corresponds to: 7.23.35 PM.png (Other Equity) & 7.23.53 PM.jpg (Cash Flow)
#---------------------------------------------------------------------
def extract_financial_report_sheet(sheet):
    """
    This sheet is a report, not a clean table. We parse it row-by-row,
    looking for a description in one column and values in others.
    """
    print(f"--- Extracting from Report Sheet: {sheet.title} ---")
    report_data = []
    # We need to define the columns for description and values
    # For 'Other Equity' sheet: Description=B, Val1=E, Val2=G
    # For 'Cash Flow' sheet: Description=B, Val1=D, Val2=E
    # This example assumes the 'Other Equity' layout
    desc_col, val1_col, val2_col = 2, 5, 7 # B, E, G

    # Iterate over a specific range of rows where data is present
    for row in sheet.iter_rows(min_row=71, max_row=91):
        description = row[desc_col - 1].value
        value1 = row[val1_col - 1].value
        value2 = row[val2_col - 1].value

        # Only capture rows that have a description
        if description:
            report_data.append({
                "Description": str(description).strip(),
                "March 31, 2024": value1,
                "March 31, 2023": value2
            })
            
    return report_data

#---------------------------------------------------------------------
# Function 3: Hybrid Extraction (Pandas for table, Openpyxl for rest)
# Corresponds to: Screenshot 2025-10-13 at 7.23.58 PM.jpg
#---------------------------------------------------------------------
def extract_hybrid_sheet(sheet, file_path):
    """
    Uses Pandas for the clean tabular part and Openpyxl for everything else.
    """
    print(f"--- Extracting from Hybrid Sheet: {sheet.title} ---")
    
    # Part A: Use Pandas for the main table
    # We specify the header row (row 18, so index 17) and the rows to skip
    try:
        print("\n[Pandas] Extracting main data table...")
        df = pd.read_excel(file_path, sheet_name=sheet.title, skiprows=18, nrows=14, usecols="B:E")
        # You might need to clean up the resulting DataFrame
        # For example, dropping rows that are entirely empty
        df.dropna(how='all', inplace=True)
        print(df)
    except Exception as e:
        print(f"Could not extract table with Pandas: {e}")

    # Part B: Use Openpyxl for non-tabular data
    print("\n[Openpyxl] Extracting surrounding text...")
    
    # Extracting the title in B17
    title = sheet['B17'].value
    print(f"Title: {title}")

    # Extracting the signature block (example for one signature)
    partner_name = sheet['B44'].value
    membership_no = sheet['B45'].value
    place = sheet['B50'].value
    
    print("\nSignature Block:")
    print(f"Partner: {partner_name}")
    print(f"Membership No: {membership_no}")
    print(f"Place: {place}")


# --- Main Execution ---
if __name__ == "__main__":
    # IMPORTANT: Replace these with the ACTUAL names of your sheets
    notes_sheet_name = 'SAP' 
    equity_sheet_name = 'Note 14 (Equity)'
    reserves_sheet_name = 'Cash Flow'

    # Example for the "Notes" sheet
    if notes_sheet_name in workbook.sheetnames:
        notes_sheet = workbook[notes_sheet_name]
        extracted_notes = extract_notes_sheet(notes_sheet)
        print(extracted_notes[:500] + "...") # Print first 500 chars
        print("\n" + "="*50 + "\n")
    else:
        print(f"Sheet '{notes_sheet_name}' not found.")

    # Example for the "Other Equity" report-style sheet
    if equity_sheet_name in workbook.sheetnames:
        equity_sheet = workbook[equity_sheet_name]
        equity_data = extract_financial_report_sheet(equity_sheet)
        for item in equity_data:
            print(item)
        print("\n" + "="*50 + "\n")
    else:
         print(f"Sheet '{equity_sheet_name}' not found.")

    # Example for the "Reserves and Surplus" hybrid sheet
    if reserves_sheet_name in workbook.sheetnames:
        reserves_sheet = workbook[reserves_sheet_name]
        extract_hybrid_sheet(reserves_sheet, file_path)
    else:
        print(f"Sheet '{reserves_sheet_name}' not found.")

In [ ]:
import fitz  # PyMuPDF
import os

def highlight_bbox_on_pdf(input_pdf, output_pdf, bbox, page_num=0):
    """
    Highlights a specified bounding box on a PDF page.

    Args:
        input_pdf (str): Path to the input PDF file.
        output_pdf (str): Path to save the modified PDF file.
        bbox (list): A list of 4 coordinates [x0, y0, x1, y1].
        page_num (int): The 0-indexed page number to highlight.
    """
    try:
        # Open the source PDF
        doc = fitz.open(input_pdf)
        if page_num >= len(doc):
            print(f"Error: Invalid page number. The PDF has {len(doc)} pages.")
            return

        # Select the page
        page = doc[page_num]

        # Define the rectangle using the bounding box
        highlight_rect = fitz.Rect(bbox)

        # Add a highlight annotation
        annot = page.add_highlight_annot(highlight_rect)

        # (Optional) Customize the highlight color and opacity
        annot.set_colors(stroke=(1, 1, 0))  # Yellow
        annot.set_opacity(0.4)             # 40% transparent
        annot.update()

        # Save the changes to the output file
        doc.save(output_pdf, garbage=4, deflate=True, clean=True)
        print(f"✅ Successfully created '{output_pdf}'")

    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        if 'doc' in locals() and doc:
            doc.close()

In [ ]:
if __name__ == "__main__":
    # 1. Define your parameters
    input_file = "./IRIS/ComparisonScriptsOctober/SFS XBRL PDF_53-54.pdf"
    output_file = "highlighted_output.pdf"
    page_number = 0  # First page
    bounding_box = [114, 359, 575, 373]

    # 2. Create a sample PDF to make the script runnable

    # 3. Run the highlight function
    highlight_bbox_on_pdf(input_file, output_file, bounding_box, page_number)